In [ ]:
import pyodbc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

# prep

In [ ]:
# Data prep
query = f"""

SELECT
  CASE WHEN borrower_id < lender_id THEN borrower_id ELSE lender_id END AS a,
  CASE WHEN borrower_id < lender_id THEN lender_id ELSE borrower_id END AS b
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state
WHERE intragroup = 1
GROUP BY
  CASE WHEN borrower_id < lender_id THEN borrower_id ELSE lender_id END,
  CASE WHEN borrower_id < lender_id THEN lender_id ELSE borrower_id END;


"""

df_edges = pd.read_sql_query(query, cnxn)

In [ ]:
import pandas as pd

# edges: columns ['a','b']
edges = df_edges.values.tolist()

parent = {}
rank = {}

def find(x):
    if parent.setdefault(x, x) != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(x, y):
    rx, ry = find(x), find(y)
    if rx == ry: return
    rxr, ryr = rank.setdefault(rx,0), rank.setdefault(ry,0)
    if rxr < ryr:
        parent[rx] = ry
    elif rxr > ryr:
        parent[ry] = rx
    else:
        parent[ry] = rx
        rank[rx] = rxr + 1

for a,b in edges:
    union(a,b)

# canonical id = min-LEI per component (stable label)
from collections import defaultdict
comps = defaultdict(list)
for x in parent.keys():
    comps[find(x)].append(x)

canon = {rep: min(members) for rep, members in comps.items()}
mapping = []
for x in parent.keys():
    rep = find(x)
    mapping.append((x, canon[rep]))

df_mapping_cc = pd.DataFrame(mapping, columns=["lei","group_id_synth"]).sort_values(["group_id_synth","lei"])


In [ ]:
# Data prep
query = f"""

SELECT security_isin
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state f
WHERE security_type IN ('GOVS', 'FIDE')
AND assttp_scty_issr_sector_riad = 'S1311'
AND business_date >= '2023-01-01' 
AND gnlcoll = 'SPEC'
GROUP BY security_isin
HAVING COUNT(DISTINCT security_type) = 2

"""

govs = pd.read_sql_query(query, cnxn)

In [ ]:
unique_govs = tuple(govs['security_isin'])

In [ ]:
foreign_bg = ('0W1U67PTV5WY3WYWKD79',
'2138008P9NOMBRMROI73',
'213800A9GT65GAES2V60',
'213800G8QEXN34A2YG53',
'213800GVD8L87R18CT98',
'213800IBT39XQ9C4CP71',
'213800NW35DTWHTMX505',
'213800RZ3GCUXMBGYN59',
'2IGI19DL77OX0HC3ZE78',
'4PQUHN3JPFGFNF3BB653',
'4ZHCHI4KYZG2WVRT8631',
'5299007QVIQ7IO64NX37',
'5493000IQQ05Y25L0V92',
'5493000YPN33HF74SN02',
'54930010P7BUGOECPI58',
'5493002XYZZ0CGQ6CB58',
'54930040QPHHWGT1J432',
'54930050SE0SM7CM2G07',
'5493006PWI2H6PX25403',
'5493007VSMFZCPV1NB83',
'5493008GNQHVI377MY19',
'5493009NLZXZGJDOPC94',
'549300BKWHXYEXPV0328',
'549300FH0WJAPEHTIQ77',
'549300HU9EWFS3CNX640',
'549300KP56LL8NKKFL47',
'549300RSY622D5TQWS42',
'549300SXSTGQY3EA1B18',
'549300WDT1HWUMTUW770',
'571474TGEMMWANRLN572',
'724500AAT1DK36059L16',
'9676007O0UF5YB3QPR03',
'BWS7DNS2Z4NPKPNYKL75',
'HV5W8PGLJ127N2SFSM23'
)

In [ ]:
euro_area_bg = ('0IKLU6X1B10WK7X42C15',
'1VUV7VQFKUOQSJ21A208',
'2138001YBO7CJNQOEE37',
'21380027LW8AF6I1WA03',
'2138003Z5ZVN16GFYV70',
'2138004VZX8CSGPTDX68',
'21380073P7J4PAD91E29',
'213800AGKVL18YKQSV51',
'213800BWHAS44J2C1B28',
'213800D4LHBCXXEEE235',
'213800DBQIB6VBNU5C64',
'213800FKGFR7Q2ACLS83',
'213800G63T4ER4MSVR22',
'213800HV6TP2I5A6MW58',
'213800I92TAU7I3FP232',
'213800KGF4EFNUQKAT69',
'213800OOQOSULB37T658',
'213800ZIGVOZ992FNQ85',
'222100D7H9VRJEH7DU25',
'222100M2PU043YB7YQ06',
'23B6332KMR0JIZLJG565',
'2534006G7F7F1TFC9T77',
'2534006HF1L4YF10UD91',
'253400N2R0RF14JQ0060',
'2549002MVYWWDX54IB83',
'259400QHDOZWMJ103294',
'259400YLRTOBISHBVX41',
'2W8N8UU78PMDQKZENC08',
'31570010000000036567',
'315700GBLUBZ50S45F53',
'315700GXRKZ452JF2U13',
'3M5E1GQGKL17HI6CPN30',
'3U8WV1YX2VMUHH7Z1Q21',
'48510000156ESYOBV122',
'52965FONQ5NZKP0WZL45',
'52990002O5KK6XOGJ020',
'52990010C4NK412ZL440',
'5299004TE2DYMKEAM814',
'5299005UJX6K7BQKV086',
'52990080NNXXLC14OC65',
'529900AQBND3S6YJLY83',
'529900C214QOT3ZYD838',
'529900C4RSSBWXBSY931',
'529900E1WHT64CB20277',
'529900FWTU88V844B672',
'529900GGYMNGRQTDOO93',
'529900K16YGKC8BES892',
'529900T32UL0CP1FZA06',
'529900UKZBMDBDZIXD62',
'529900VA5CNBWXAONR25',
'549300298FD7AS4PPU70',
'54930056IRBXK0Q1FP96',
'549300685QG7DJS55M76',
'5493007RT80TMKY7TO30',
'549300ABE4K96QOCEH37',
'549300CQ9NLEHMRCU505',
'549300DV870NBWY5W279',
'549300DY78U4CMKNHE48',
'549300DYPOFMXOR7XM56',
'549300FOF121DSRG5867',
'549300FR956J8UJDWQ78',
'549300GOF5A5DHWJLV03',
'549300GRXFI7D6PNEA68',
'549300L7YCATGO57ZE10',
'549300LYFYVPUCG6SY25',
'549300NC3SZTETC10349',
'549300NEBDPH0ZXIF850',
'549300Z6OB1D4ZUBD145',
'63540061DPCBNMCGRY22',
'635400CE9HHFB55PEY43',
'635400GQWXFJDCQXW612',
'635400LNHEPZBRNB5D58',
'635400LRAHYBRUZCIH13',
'724500A11NIP5HCVF984',
'815600154F8F91CF6B05',
'8156002070DA4DCBFF31',
'81560027D07F9BDB8436',
'81560038903FF9FA8F80',
'815600522538355AE429',
'8156007395B20763EB44',
'8156009F40F6523F7022',
'815600A32DA05F693F24',
'8EFE15WY4PBBKG6GZI21',
'95980020140005184148',
'9598009X93GQBNHF1L85',
'959800T0J9ADL4GYSS41',
'9695000O0HNLFNR0FB30',
'969500DCEVPV6UIYK220',
'969500QLO3GN6GUB4P61',
'FI6C7E5PBUB3F9K43B44',
'GP5DT10VX1QRQUKVBK64',
'J4CP7MHCXR8DAQMKIL78',
'NNVPP80YIZGEY2314M97',
'RRAN7P32P0W0YY4XQW79',
'SI5RG2M0WQQLZCXKRM20'
)

In [ ]:
foreign_entity = tuple(df_mapping_cc.loc[df_mapping_cc['group_id_synth'].isin(foreign_bg), 'lei'].unique())
euro_area_entity = tuple(df_mapping_cc.loc[df_mapping_cc['group_id_synth'].isin(euro_area_bg), 'lei'].unique())

# columns

Field list of the table, needed later to find the maturity and rate type fields for the tenor split. If LIMIT is not accepted use TOP 5.

In [ ]:
cols = pd.read_sql_query("SELECT * FROM xlab_ecb_prj_sftds_cb_common.hermesf_state LIMIT 5", cnxn)
list(cols.columns)

# wedge

Euro area subsidiaries of foreign groups, EUR, EGB collateral, one row per subsidiary, bond and day. Four legs with volume weighted rates.

Direction in (bond out to the affiliate). The subsidiary lends cash in the CCP and borrows cash intragroup. `chain_in = min(cl_lend_vol, in_borr_vol)`, `wedge_in = cl_lend_rate - in_borr_rate`.

Direction out (bond in from the affiliate). The subsidiary lends cash intragroup and borrows cash in the CCP. `chain_out = min(cl_borr_vol, in_lend_vol)`, `wedge_out = in_lend_rate - cl_borr_rate`.

Both wedges are in basis points and positive when the subsidiary keeps a margin. `carry` pools the two, weighted by matched volume. `tbl` is `hermesf_state` for the outstanding book and `hermesf_flow` for new trades only.

In [ ]:
tbl = 'hermesf_state'
out = 'Data\\wedge_eur_state.csv'
ea = "('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')"

_coll = f"""
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN {ea}
     AND s.gnlcoll = 'SPEC' AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN {ea}
     AND s.gnlcoll = 'SPEC' AND s.security_isin IN {unique_govs})
  )
"""

_rate = """sum(CASE WHEN repo_rate IS NOT NULL THEN repo_rate*nominal_value END)/sum(CASE WHEN repo_rate IS NOT NULL THEN nominal_value END)"""
_cov = """sum(CASE WHEN repo_rate IS NOT NULL THEN nominal_value END)/sum(nominal_value)"""

In [ ]:
# Cleared lending, the subsidiary sources the bond in the CCP
q_cl_lend = f"""
SELECT business_date, security_isin, lender_id as entity_id,
sum(nominal_value)/1e9 as cl_lend_vol, {_rate} as cl_lend_rate, {_cov} as cl_lend_cov
FROM xlab_ecb_prj_sftds_cb_common.{tbl} s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR' AND intragroup = 0 AND central_clearing = 'cleared'
AND lender_id IN {foreign_entity}
AND lender_country_residence IN {ea}
{_coll}
GROUP BY business_date, security_isin, lender_id
"""

# Cleared borrowing, the subsidiary places the bond in the CCP
q_cl_borr = f"""
SELECT business_date, security_isin, borrower_id as entity_id,
sum(nominal_value)/1e9 as cl_borr_vol, {_rate} as cl_borr_rate, {_cov} as cl_borr_cov
FROM xlab_ecb_prj_sftds_cb_common.{tbl} s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR' AND intragroup = 0 AND central_clearing = 'cleared'
AND borrower_id IN {foreign_entity}
AND borrower_country_residence IN {ea}
{_coll}
GROUP BY business_date, security_isin, borrower_id
"""

# Intragroup lending, the subsidiary receives the bond from a non euro area affiliate
q_in_lend = f"""
SELECT business_date, security_isin, lender_id as entity_id,
sum(nominal_value)/1e9 as in_lend_vol, {_rate} as in_lend_rate, {_cov} as in_lend_cov
FROM xlab_ecb_prj_sftds_cb_common.{tbl} s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR' AND intragroup = 1 AND central_clearing = 'non-cleared'
AND lender_id IN {foreign_entity}
AND lender_country_residence IN {ea}
AND borrower_country_residence NOT IN {ea}
{_coll}
GROUP BY business_date, security_isin, lender_id
"""

# Intragroup borrowing, the subsidiary passes the bond to a non euro area affiliate
q_in_borr = f"""
SELECT business_date, security_isin, borrower_id as entity_id,
sum(nominal_value)/1e9 as in_borr_vol, {_rate} as in_borr_rate, {_cov} as in_borr_cov
FROM xlab_ecb_prj_sftds_cb_common.{tbl} s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR' AND intragroup = 1 AND central_clearing = 'non-cleared'
AND borrower_id IN {foreign_entity}
AND borrower_country_residence IN {ea}
AND lender_country_residence NOT IN {ea}
{_coll}
GROUP BY business_date, security_isin, borrower_id
"""

cl_lend = pd.read_sql_query(q_cl_lend, cnxn)
cl_borr = pd.read_sql_query(q_cl_borr, cnxn)
in_lend = pd.read_sql_query(q_in_lend, cnxn)
in_borr = pd.read_sql_query(q_in_borr, cnxn)

In [ ]:
k = ['business_date', 'security_isin', 'entity_id']
df = cl_lend.merge(cl_borr, on=k, how='outer').merge(in_lend, on=k, how='outer').merge(in_borr, on=k, how='outer')
for c in ['cl_lend_vol', 'cl_borr_vol', 'in_lend_vol', 'in_borr_vol']:
    df[c] = df[c].fillna(0)

# keep subsidiary-bond-days with some intragroup activity, cleared-only rows carry no internal price
df = df[(df['in_lend_vol'] > 0) | (df['in_borr_vol'] > 0)].copy()

df['chain_in'] = np.minimum(df['cl_lend_vol'], df['in_borr_vol'])
df['chain_out'] = np.minimum(df['cl_borr_vol'], df['in_lend_vol'])

df['wedge_in'] = (df['cl_lend_rate'] - df['in_borr_rate'])*100
df['wedge_out'] = (df['in_lend_rate'] - df['cl_borr_rate'])*100
df.loc[df['chain_in'] == 0, 'wedge_in'] = np.nan
df.loc[df['chain_out'] == 0, 'wedge_out'] = np.nan

a = df['chain_in'].where(df['wedge_in'].notna(), 0)
b = df['chain_out'].where(df['wedge_out'].notna(), 0)
df['carry'] = (df['wedge_in'].fillna(0)*a + df['wedge_out'].fillna(0)*b)/(a + b)

In [ ]:
# Market cleared rate of the bond, all banks, volume weighted
q_mkt = f"""
SELECT business_date, security_isin,
sum(nominal_value)/1e9 as mkt_vol,
sum(repo_rate*nominal_value)/sum(nominal_value) as mkt_rate
FROM xlab_ecb_prj_sftds_cb_common.{tbl} s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR' AND intragroup = 0 AND central_clearing = 'cleared'
AND repo_rate IS NOT NULL
{_coll}
GROUP BY business_date, security_isin
"""
mkt = pd.read_sql_query(q_mkt, cnxn)

df = df.merge(mkt, on=['business_date', 'security_isin'], how='left')

In [ ]:
estr = pd.read_csv('ESTR.csv')
estr['Effective Date'] = pd.to_datetime(estr['DATE'], format='%d/%m/%Y')
df['business_date'] = pd.to_datetime(df['business_date'])
df = df.merge(estr[['Effective Date', 'ESTR']], left_on='business_date', right_on='Effective Date', how='left')
df = df.drop(columns='Effective Date')
df['special'] = (df['ESTR'] - df['mkt_rate'])*100

In [ ]:
# group id and number of subsidiaries and groups running a chain in the bond that day
df = df.merge(df_mapping_cc.rename(columns={'lei': 'entity_id', 'group_id_synth': 'group_id'}), on='entity_id', how='left')

act = df[(df['chain_in'] > 0) | (df['chain_out'] > 0)]
n = act.groupby(['business_date', 'security_isin'], as_index=False).agg(n_entities=('entity_id', 'nunique'), n_groups=('group_id', 'nunique'))
df = df.merge(n, on=['business_date', 'security_isin'], how='left')
df['n_entities'] = df['n_entities'].fillna(0)
df['n_groups'] = df['n_groups'].fillna(0)

# direct hedge fund route

Bond-day rates on the direct route, euro area banks of euro area groups against Cayman funds. `hf_s` is the fund lending cash, so the fund borrows the bond (short). `hf_l` is the fund borrowing cash (long financing). Pooled across banks, merged onto the subsidiary-bond-day rows as a benchmark for the wedge.

In [ ]:
q_hf_s = f"""
SELECT business_date, security_isin,
sum(nominal_value)/1e9 as hf_s_vol,
sum(repo_rate*nominal_value)/sum(nominal_value) as hf_s_rate
FROM xlab_ecb_prj_sftds_cb_common.{tbl} s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR' AND intragroup = 0 AND central_clearing = 'non-cleared'
AND borrower_id IN {euro_area_entity}
AND borrower_country_residence IN {ea}
AND lender_country_residence = 'KY'
AND repo_rate IS NOT NULL
{_coll}
GROUP BY business_date, security_isin
"""

q_hf_l = f"""
SELECT business_date, security_isin,
sum(nominal_value)/1e9 as hf_l_vol,
sum(repo_rate*nominal_value)/sum(nominal_value) as hf_l_rate
FROM xlab_ecb_prj_sftds_cb_common.{tbl} s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR' AND intragroup = 0 AND central_clearing = 'non-cleared'
AND lender_id IN {euro_area_entity}
AND lender_country_residence IN {ea}
AND borrower_country_residence = 'KY'
AND repo_rate IS NOT NULL
{_coll}
GROUP BY business_date, security_isin
"""

hf_s = pd.read_sql_query(q_hf_s, cnxn)
hf_l = pd.read_sql_query(q_hf_l, cnxn)
hf_s['business_date'] = pd.to_datetime(hf_s['business_date'])
hf_l['business_date'] = pd.to_datetime(hf_l['business_date'])

df = df.merge(hf_s, on=['business_date', 'security_isin'], how='left').merge(hf_l, on=['business_date', 'security_isin'], how='left')
df['hf_s_vol'] = df['hf_s_vol'].fillna(0)
df['hf_l_vol'] = df['hf_l_vol'].fillna(0)

In [ ]:
df.to_csv(out, index=False)

# checks

Rate coverage on the intragroup legs decides whether the price strand is alive. Entity and group counts are for the confidentiality rule.

In [ ]:
df[['cl_lend_cov', 'cl_borr_cov', 'in_lend_cov', 'in_borr_cov']].describe()

In [ ]:
df[['wedge_in', 'wedge_out', 'carry']].describe()

In [ ]:
df[['chain_in', 'chain_out']].sum()

In [ ]:
df['year'] = df['business_date'].dt.year
df[(df['chain_in'] > 0) | (df['chain_out'] > 0)].groupby('year').agg(entities=('entity_id', 'nunique'), groups=('group_id', 'nunique'), rows=('security_isin', 'size'))

In [ ]:
(df.loc[(df['chain_in'] > 0) | (df['chain_out'] > 0), 'n_groups'] >= 3).mean()